# 기준 요청과 현재 변화 조건의 입력 분포 비교

## 이번 질문

기준 모델의 `high_risk` 예측 비율이 늘었을 때 입력 조건 변화가 원인 후보인지 확인합니다. 정답 없는 운영 표본의 분포를 비교하되 모델 성능 저하나 단일 원인을 확정하지 않습니다.

## 먼저 예상

나이, 평균 심박수, 마지막 젖산값과 최소 GCS가 `current-shift`에서 어느 방향으로 움직일지 먼저 적습니다. 평균만 볼 때 놓칠 수 있는 변화도 예상합니다.

## 실행과 관측

### 1. 비교할 표본과 변환 확인

In [ ]:
from pathlib import Path

import pandas as pd
import yaml

ROOT = next(
    parent
    for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "pyproject.toml").exists()
    and (parent / "configs/traffic/scenarios.yaml").exists()
)
OPERATIONAL_DATA = ROOT / "data/splits/physionet-2012/revisions/v2/datasets/operational.csv"
SCENARIOS = ROOT / "configs/traffic/scenarios.yaml"

baseline = pd.read_csv(OPERATIONAL_DATA)
scenario_config = yaml.safe_load(SCENARIOS.read_text(encoding="utf-8"))
transforms = scenario_config["scenarios"]["current-shift"]["transforms"]

print(f"rows={len(baseline)}, columns={len(baseline.columns)}")
print(f"target_present={'target' in baseline.columns}")
pd.DataFrame(transforms).T

### 2. 같은 100건에 변화 조건 적용

운영 요청 표본에는 `target`이 없어야 합니다. 정답이 없는 자료로 새 재현율이나 정밀도를 계산하지 않기 위해서입니다. `current-shift`는 나이, 평균 심박수, 마지막 젖산값, 최소 GCS에 정해진 변환만 적용합니다.

In [ ]:
current_shift = baseline.copy()
for feature, rule in transforms.items():
    values = current_shift[feature].astype(float)
    values = values * float(rule.get("multiply", 1.0))
    values = values + float(rule.get("add", 0.0))
    if "minimum" in rule:
        values = values.clip(lower=float(rule["minimum"]))
    if "maximum" in rule:
        values = values.clip(upper=float(rule["maximum"]))
    current_shift[feature] = values

assert "target" not in baseline.columns
assert baseline["record_id"].equals(current_shift["record_id"])
assert len(baseline) == len(current_shift)
print("동일한 100개 요청에 설정된 변환만 적용했습니다.")

### 3. 비교 조건과 분모 확인

두 표본의 레코드와 나머지 특성을 그대로 둔 채 네 특성만 바뀌었는지 확인합니다. 비교 조건을 고정해야 예측 분포가 달라졌을 때 입력 변화와 모델 교체의 효과를 섞지 않을 수 있습니다.

In [ ]:
rows: list[dict[str, float | int | str]] = []
for feature in transforms:
    baseline_values = baseline[feature]
    shifted_values = current_shift[feature]
    rows.append(
        {
            "feature": feature,
            "observed_rows": int(baseline_values.notna().sum()),
            "missing_rows": int(baseline_values.isna().sum()),
            "baseline_mean": round(float(baseline_values.mean()), 3),
            "shift_mean": round(float(shifted_values.mean()), 3),
            "baseline_median": round(float(baseline_values.median()), 3),
            "shift_median": round(float(shifted_values.median()), 3),
            "baseline_p05": round(float(baseline_values.quantile(0.05)), 3),
            "shift_p05": round(float(shifted_values.quantile(0.05)), 3),
            "baseline_p95": round(float(baseline_values.quantile(0.95)), 3),
            "shift_p95": round(float(shifted_values.quantile(0.95)), 3),
        }
    )

comparison = pd.DataFrame(rows)
comparison

## 해석과 기록

### 4. 분포 요약 해석

`observed_rows`와 `missing_rows`는 각 요약값의 분모를 설명합니다. 평균과 중앙값만 보면 꼬리 구간의 변화를 놓칠 수 있으므로 P05와 P95도 함께 비교합니다.

설정한 방향으로 여러 요약값이 움직이면 입력 조건 변화라는 원인 후보는 강화됩니다. 다만 이 표는 준비된 100건을 재현한 결과입니다. 실제 대상 환경에서도 같은 변화가 있었다고 말하려면 같은 모델 정보, 시나리오와 시간 범위의 운영 근거가 더 필요합니다.

## 결과 점검

In [ ]:
changed_columns = [
    column
    for column in baseline.columns
    if not baseline[column].equals(current_shift[column])
]
expected_columns = list(transforms)
print(f"changed_columns={changed_columns}")
print(f"expected_columns={expected_columns}")
assert set(changed_columns) == set(expected_columns)
assert comparison["observed_rows"].add(comparison["missing_rows"]).eq(len(baseline)).all()

evidence = {
    "scope": "prepared_course_evidence",
    "row_count": len(baseline),
    "same_base_rows": baseline["record_id"].equals(current_shift["record_id"]),
    "changed_columns": changed_columns,
    "fact": "설정된 네 특성의 입력 분포가 기준 요청과 달라졌다",
    "cause_candidate": "입력 조건 변화: 강화",
    "not_proven": [
        "모델 성능 저하",
        "실제 운영 환경에서 같은 변화가 발생함",
        "예측 비율 증가의 단일 원인",
    ],
    "next_evidence": "같은 모델 정보, Scenario와 시간 범위의 점수, 예측, 요청 기록",
}
evidence

## 다음 확인

같은 모델 정보와 시간 범위에서 시나리오별 점수, 예측 지표, 로그와 실제 요청 trace를 확인합니다. 준비된 분포 비교를 실제 대상 환경 관측으로 바꾸지 않습니다.